Tự zip thư mục project (thư mục chứa `requirements.txt`) rồi upload thành 1 Kaggle Dataset mới, gắn vào Kaggle
qua **Add Input**

**Trước khi chạy:** trong Kaggle notebook editor, bật **GPU** và **Internet** ở mục Settings


## 1. Lấy code


In [ ]:
REPO_URL = None  # vd "https://github.com/<ban>/<repo>.git"

import glob
import os
import shutil
import zipfile

PROJECT_DIR = "/kaggle/working/pkg-halluc"


def looks_like_project(d):
    return os.path.isfile(os.path.join(d, "requirements.txt")) and os.path.isdir(
        os.path.join(d, "scripts")
    )


if looks_like_project(PROJECT_DIR):
    print("Đã có sẵn tại", PROJECT_DIR)
elif REPO_URL:
    !git clone --depth 1 "$REPO_URL" "$PROJECT_DIR"
else:
    zip_candidates = glob.glob("/kaggle/input/**/pkg-halluc.zip", recursive=True) or glob.glob(
        "/kaggle/input/**/*.zip", recursive=True
    )
    dir_candidates = [
        os.path.dirname(p) for p in glob.glob("/kaggle/input/**/requirements.txt", recursive=True)
    ]
    assert zip_candidates or dir_candidates, (
        "Không tìm thấy project trong /kaggle/input, và REPO_URL đang trống. "
        "Hãy zip thư mục project (thư mục chứa requirements.txt) trên máy bạn, "
        "upload thành 1 Kaggle Dataset mới, gắn vào notebook này qua "
        "'Add Input' rồi chạy lại cell này."
    )
    if zip_candidates:
        print("Tìm thấy file zip:", zip_candidates[0])
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(PROJECT_DIR)
        # nếu zip có 1 thư mục bọc ngoài (vd "project/"), đưa nội dung lên trên
        entries = [e for e in os.listdir(PROJECT_DIR) if not e.startswith(".")]
        if len(entries) == 1 and os.path.isdir(os.path.join(PROJECT_DIR, entries[0])):
            inner = os.path.join(PROJECT_DIR, entries[0])
            if looks_like_project(inner):
                for item in os.listdir(inner):
                    shutil.move(os.path.join(inner, item), os.path.join(PROJECT_DIR, item))
                os.rmdir(inner)
    else:
        print("Tìm thấy dataset đã giải nén sẵn:", dir_candidates[0])
        shutil.copytree(dir_candidates[0], PROJECT_DIR, dirs_exist_ok=True)

assert looks_like_project(PROJECT_DIR), f"{PROJECT_DIR} không giống project (thiếu requirements.txt)"
%cd {PROJECT_DIR}
!chmod +x scripts/*.sh


## 2. Cài đặt


In [ ]:
!pip install -r requirements.txt -q


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ -- bật GPU trong Settings!")


## 3. Chọn model

Mặc định chạy bằng tham số có sẵn trong các script, chỉ cần đặt `MODEL`.
Muốn nạp cả bộ tham số từ file JSON (tuỳ chọn) thì đặt `CONFIG`, khi đó `MODEL` lấy từ file.


In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"  # tên preset hoặc id HF
CONFIG = None  # vd "configs/smoke_test.json", None = dùng tham số mặc định

import json, os, sys

sys.path.insert(0, os.getcwd())  # để import được pkg_halluc
from pkg_halluc.common.model_presets import resolve_model_name

if CONFIG:
    print(open(CONFIG, encoding="utf-8").read())
    with open(CONFIG, encoding="utf-8") as f:
        MODEL = json.load(f)["model_name"]
MODEL = resolve_model_name(MODEL)

ARGS = f'--config "{CONFIG}"' if CONFIG else f'--model "{MODEL}"'
EVAL_ARGS = f'--config "{CONFIG}"' if CONFIG else ""
print("MODEL =", MODEL, "| ARGS =", ARGS)


## 4. Chạy pipeline


In [ ]:
!bash scripts/download_model.sh {ARGS}


In [ ]:
!bash scripts/build_data.sh {ARGS}


In [ ]:
!bash scripts/train_ga.sh {ARGS}
!bash scripts/train_npo.sh {ARGS}


In [ ]:
!bash scripts/train_ga_plain.sh {ARGS}
!bash scripts/train_npo_plain.sh {ARGS}


In [ ]:
MODEL_BASENAME = MODEL.split("/")[-1]
BASE_PATH = f"models/{MODEL}"
GA_PATH = f"checkpoints/{MODEL_BASENAME}_ga"
NPO_PATH = f"checkpoints/{MODEL_BASENAME}_npo"
GA_PLAIN_PATH = f"checkpoints/{MODEL_BASENAME}_ga_plain"
NPO_PLAIN_PATH = f"checkpoints/{MODEL_BASENAME}_npo_plain"

!bash scripts/eval.sh {EVAL_ARGS} --tag base      --model-path "{BASE_PATH}"
!bash scripts/eval.sh {EVAL_ARGS} --tag ga        --model-path "{GA_PATH}"
!bash scripts/eval.sh {EVAL_ARGS} --tag npo       --model-path "{NPO_PATH}"
!bash scripts/eval.sh {EVAL_ARGS} --tag ga_plain  --model-path "{GA_PLAIN_PATH}"
!bash scripts/eval.sh {EVAL_ARGS} --tag npo_plain --model-path "{NPO_PLAIN_PATH}"


## 5. Report


In [ ]:
!bash scripts/report.sh --model-name "{MODEL}"


In [ ]:
import pandas as pd
display(pd.read_csv(".workdir/outputs/table1_hallucination_rate.csv"))
display(pd.read_csv(".workdir/outputs/table1_by_context.csv"))
